In [1]:
# LOCAL EXECUTION SETUP — sets cwd to repo root so relative paths work
import os
os.chdir('/Users/danilezov/work/optuna_exp/PINNacle')
print('cwd:', os.getcwd())

cwd: /Users/danilezov/work/optuna_exp/PINNacle


# PINNacle Optuna Optimizer Search — Kaggle Runner

This notebook clones the PINNacle repo from GitHub, installs dependencies,
and runs the Optuna optimizer chain search for a chosen PDE.

**Configure the cell below, then run all cells.**

In [2]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Repo URL — set to your fork/branch before running
GITHUB_REPO = "https://github.com/YOUR_USERNAME/PINNacle.git"
BRANCH = "main"  # or e.g. "optuna-experiments"

# Which PDE to optimize. Available choices:
#   burgers_1d, burgers_2d,
#   heat2d_varyingcoef, heat2d_multiscale, heat2d_complexgeometry,
#   heat2d_longtime, heatnd,
#   grayscott, kuramoto_sivashinsky,
#   poissoninv, heatinv,
#   ns2d_classic, ns2d_backstep, ns2d_longtime,
#   poisson2d_classic, poissonboltzmann2d, poisson3d_complexgeometry,
#   poisson2d_manyarea, poissonnd,
#   wave1d, wave2d_heterogeneous, wave2d_longtime
PDE_NAME = "burgers_1d"

# Value type: 'continuous' (TPE over ranges) or 'fixed' (discrete grid)
VALUE_TYPE = "continuous"

# Number of parallel Optuna workers.
# Set based on GPU memory (see pde_gpu_benchmark.csv):
#   - Kaggle T4 (~15 GB): 1-2 processes for heavy PDEs, up to 4 for light ones
#   - Kaggle P100 (~16 GB): similar
#   - Kaggle 2×T4 (~30 GB): double the above
N_PROCESSES = 1

# Trials per worker (shared Optuna study → total trials ≈ N_PROCESSES × N_TRIALS)
N_TRIALS = 1

# Wall-clock hours for Optuna.optimize per worker (Kaggle sessions ~9h)
TIMEOUT_HOURS = 2

# Eval runs with the best found chain after Optuna (set 0 to skip)
N_EVAL_RUNS = 1

# Optuna sampler
SAMPLER = "tpe"  # 'tpe' or 'random'

# ── Comet ML ──────────────────────────────────────────────────────────────────
COMET_API_KEY       = "NM8cfXp7qp88rpeXKKvf9ZIdd"
COMET_PROJECT_NAME  = "optuna-rl-kaggle"
COMET_WORKSPACE     = "florentiner"
# ──────────────────────────────────────────────────────────────────────────────

In [3]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  ({props.total_memory // 1024**2} MB)")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    print("  MPS (Apple Silicon) available — running locally on MPS.")
else:
    raise RuntimeError("No GPU found — Kaggle kernel must have GPU accelerator enabled.")

PyTorch version: 2.12.0
CUDA available: False
  MPS (Apple Silicon) available — running locally on MPS.


In [4]:
import subprocess, os

# Clone repo (skip if running locally inside the repo already)
if not os.path.exists("PINNacle") and not os.path.exists("experiments"):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, "--single-branch", GITHUB_REPO, "PINNacle"],
        check=True
    )
else:
    print("Repo already present, skipping clone.")

Repo already present, skipping clone.


In [5]:
# Navigate to repo root (skip if already there)
if os.path.exists("PINNacle") and not os.path.exists("experiments"):
    os.chdir("PINNacle")
print("Working directory:", os.getcwd())

Working directory: /Users/danilezov/work/optuna_exp/PINNacle


In [6]:
!pip3 install -r requirements.txt -q
!pip3 install comet_ml -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [7]:
# Verify deepxde imports correctly with pytorch backend
import sys
sys.path.insert(0, os.getcwd())
os.environ["DDEBACKEND"] = "pytorch"
import deepxde as dde
print("DeepXDE version:", dde.__version__)

Using backend: pytorch



DeepXDE version: 1.8.1


In [8]:
# Build the shell command to launch N_PROCESSES parallel Optuna workers
import shlex, sys

SCRIPT = f"experiments/optuna_multi_pde/{PDE_NAME}_optuna.py"
DB_PATH = f"optuna_studies/{PDE_NAME}.db"
STUDY_NAME = f"{PDE_NAME}_chain"
RESULTS_CSV = f"{PDE_NAME}.csv"
os.makedirs("optuna_studies", exist_ok=True)

# Use the current Python executable so local env / MPS support is preserved
PYTHON_BIN = sys.executable

base_cmd = [
    PYTHON_BIN, SCRIPT,
    "--db-path", DB_PATH,
    "--study-name", STUDY_NAME,
    "--results-csv", RESULTS_CSV,
    "--n-trials", str(N_TRIALS),
    "--n-eval-runs", str(N_EVAL_RUNS),
    "--timeout-hours", str(TIMEOUT_HOURS),
    "--sampler", SAMPLER,
    "--value-type", VALUE_TYPE,
    "--test-epochs", "3",
]

print("Script:", SCRIPT)
print("Command:", shlex.join(base_cmd))
print(f"Will spawn {N_PROCESSES} worker(s).")

Script: experiments/optuna_multi_pde/burgers_1d_optuna.py
Command: /Users/danilezov/work/optuna_exp/PINNacle/.venv/bin/python3 experiments/optuna_multi_pde/burgers_1d_optuna.py --db-path optuna_studies/burgers_1d.db --study-name burgers_1d_chain --results-csv burgers_1d.csv --n-trials 1 --n-eval-runs 1 --timeout-hours 2 --sampler tpe --value-type continuous --test-epochs 3
Will spawn 1 worker(s).


In [9]:
import subprocess

# Determine GPU assignment: CUDA (round-robin) or MPS (single device)
n_gpus = torch.cuda.device_count()
use_mps = (n_gpus == 0 and getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
procs = []

for i in range(N_PROCESSES):
    env = os.environ.copy()
    env["DDEBACKEND"] = "pytorch"
    if use_mps:
        env["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"  # let MPS use all available memory
        device_label = "mps"
    else:
        gpu_id = i % n_gpus
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        device_label = f"cuda:{gpu_id}"
    p = subprocess.Popen(base_cmd, env=env)
    procs.append(p)
    print(f"Worker {i} started ({device_label}, PID {p.pid})")

print(f"\nAll {N_PROCESSES} workers launched. Waiting for completion...")
for p in procs:
    p.wait()
    print(f"Worker PID {p.pid} finished with exit code {p.returncode}")

print("\nAll workers done.")

Worker 0 started (mps, PID 98659)

All 1 workers launched. Waiting for completion...


Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Traceback (most recent call last):
  File "/Users/danilezov/work/optuna_exp/PINNacle/experiments/optuna_multi_pde/burgers_1d_optuna.py", line 27, in <module>
    from optuna_trainer import run_optuna_study
  File "/Users/danilezov/work/optuna_exp/PINNacle/optuna_trainer.py", line 15, in <module>
    from deepxde.optimizers.config import set_PSO_options
ImportError: cannot import name 'set_PSO_options' from 'deepxde.optimizers.config' (/Users/danilezov/work/optuna_exp/PINNacle/.venv/lib/python3.11/site-packages/deepxde/optimizers/config.py)


Worker PID 98659 finished with exit code 1

All workers done.


In [10]:
# Show results CSV
import pandas as pd

csv_path = RESULTS_CSV
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Results from {csv_path}:")
    display(df)
else:
    print(f"No results CSV found at {csv_path}")
    for root, dirs, files in os.walk("runs_optuna"):
        for fn in files:
            if fn == "results.csv":
                p = os.path.join(root, fn)
                df = pd.read_csv(p)
                print(f"\nFound: {p}")
                display(df.head(20))
                break

Results from burgers_1d.csv:


,phase,run_id,mse,rmse,brmse,trajectory_json
0,optuna_best,0,NaN,NaN,NaN,"[{""optimizer"": ""LBFGS"", ""lr"": 0.89693386956629..."
1,eval,0,inf,inf,inf,"[{""optimizer"": ""LBFGS"", ""lr"": 0.89693386956629..."
2,eval_summary,mean,NaN,NaN,NaN,"[{""optimizer"": ""LBFGS"", ""lr"": 0.89693386956629..."


In [11]:
# Show Optuna study summary
import optuna

storage_url = f"sqlite:///{os.path.abspath(DB_PATH)}"
try:
    study = optuna.load_study(study_name=STUDY_NAME, storage=storage_url)
    print(f"Study: {STUDY_NAME}")
    print(f"Total trials: {len(study.trials)}")
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    print(f"Completed: {len(completed)}")
    if completed:
        print(f"Best trial #: {study.best_trial.number}")
        print(f"Best objective (rmse + brmse): {study.best_trial.value:.6f}")
        print(f"Best params:")
        for k, v in study.best_trial.params.items():
            print(f"  {k}: {v}")
except Exception as e:
    print(f"Could not load study: {e}")

/Users/danilezov/work/optuna_exp/PINNacle/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Study: burgers_1d_chain
Total trials: 1
Completed: 1
Best trial #: 0
Best objective (rmse + brmse): inf
Best params:
  step_0_type: LBFGS
  step_0_lr: 0.8854820772958851
  step_0_epochs: 3
  step_1_type: LBFGS
  step_1_lr: 0.08893802295994357
  step_1_epochs: 3
  step_2_type: Adam
  step_2_lr: 0.2883698571194301
  step_2_epochs: 3
  step_3_type: Adam
  step_3_lr: 0.6144323748886419
  step_3_epochs: 3
  step_4_type: LBFGS
  step_4_lr: 0.8789762407364188
  step_4_epochs: 3


In [12]:
# ── Push results to Comet ML ──────────────────────────────────────────────────
# Metrics logged from the results CSV (phase=optuna_best / eval / eval_summary):
#   rmse_op    = operator/domain RMSE  = sqrt(domain MSE)
#   rmse_bnd   = boundary RMSE         = sqrt(boundary MSE)
#   rmse_total = rmse_op + rmse_bnd    (the Optuna objective)
#   mse_op     = rmse_op^2             (domain MSE)
#   mse_bnd    = rmse_bnd^2            (boundary MSE)
#   mse_total  = mse_op + mse_bnd
#
# Note: L2RE = rmse_op / ||ref||_rms requires the reference solution norm which
# is computed inside TesterCallback at training time (not stored in the CSV).
# It is equivalent to rmse_op when the reference is unit-normalized.
#
# Hyperparams: pde, value_type, n_processes, sampler, best chain step config
# ──────────────────────────────────────────────────────────────────────────────
import math, json
import numpy as _np
import pandas as _pd
import optuna as _optuna
from comet_ml import start as comet_start

EXPERIMENT_NAME = f"{PDE_NAME}_{VALUE_TYPE}"

experiment = comet_start(
    api_key=COMET_API_KEY,
    project_name=COMET_PROJECT_NAME,
    workspace=COMET_WORKSPACE,
)
experiment.set_name(EXPERIMENT_NAME)

def _to_float(v):
    """Safe float conversion; returns None for NaN/Inf/missing."""
    try:
        f = float(v)
        return f if math.isfinite(f) else None
    except (TypeError, ValueError):
        return None

try:
    # ── Study metadata ──────────────────────────────────────────────────
    _storage_url = f"sqlite:///{os.path.abspath(DB_PATH)}"
    _study = _optuna.load_study(study_name=STUDY_NAME, storage=_storage_url)
    _completed = [t for t in _study.trials if t.state == _optuna.trial.TrialState.COMPLETE]

    # ── Hyperparams ───────────────────────────────────────────────────
    hparams = {
        "pde": PDE_NAME,
        "value_type": VALUE_TYPE,
        "n_processes": N_PROCESSES,
        "n_trials_per_worker": N_TRIALS,
        "timeout_hours": TIMEOUT_HOURS,
        "n_eval_runs": N_EVAL_RUNS,
        "sampler": SAMPLER,
        "total_trials_in_study": len(_study.trials),
        "completed_trials": len(_completed),
    }
    if _completed:
        _best = _study.best_trial
        hparams["best_trial_number"] = _best.number
        hparams["best_objective_rmse_plus_brmse"] = _best.value
        # Per-step chain params (step_0_type, step_0_lr, step_0_epochs, ...)
        for k, v in _best.params.items():
            hparams[f"best_{k}"] = v
    experiment.log_parameters(hparams)

    # ── Metrics from results CSV ────────────────────────────────────────
    if os.path.exists(RESULTS_CSV):
        _df = _pd.read_csv(RESULTS_CSV)

        # optuna_best row: the single best trial's final-stage metrics
        _best_row = _df[_df["phase"] == "optuna_best"]
        if not _best_row.empty:
            _r = _best_row.iloc[0]
            _ro = _to_float(_r.get("rmse"))   # operator RMSE
            _rb = _to_float(_r.get("brmse"))  # boundary RMSE
            _mo = _to_float(_r.get("mse"))    # operator MSE (= rmse^2)
            if _ro is not None:
                _mso = _mo if _mo is not None else _ro**2
                _m = {"best_rmse_op": _ro, "best_mse_op": _mso}
                if _rb is not None:
                    _m.update({
                        "best_rmse_bnd": _rb,
                        "best_mse_bnd": _rb**2,
                        "best_rmse_op_plus_bnd": _ro + _rb,
                        "best_mse_op_plus_bnd": _mso + _rb**2,
                    })
                experiment.log_metrics(_m)

        # eval rows: per-run metrics (step = run index 0..N_EVAL_RUNS-1)
        for _, _row in _df[_df["phase"] == "eval"].iterrows():
            _step = int(float(_row["run_id"])) if _pd.notna(_row.get("run_id")) else None
            _ro = _to_float(_row.get("rmse"))
            _rb = _to_float(_row.get("brmse"))
            _mo = _to_float(_row.get("mse"))
            if _ro is not None:
                _mso = _mo if _mo is not None else _ro**2
                _m = {"eval_rmse_op": _ro, "eval_mse_op": _mso}
                if _rb is not None:
                    _m.update({
                        "eval_rmse_bnd": _rb,
                        "eval_mse_bnd": _rb**2,
                        "eval_rmse_total": _ro + _rb,
                        "eval_mse_total": _mso + _rb**2,
                    })
                experiment.log_metrics(_m, step=_step)

        # eval_summary row: mean metrics across all eval runs (primary summary)
        _summary = _df[_df["phase"] == "eval_summary"]
        if not _summary.empty:
            _r = _summary.iloc[0]
            _ro = _to_float(_r.get("rmse"))
            _rb = _to_float(_r.get("brmse"))
            _mo = _to_float(_r.get("mse"))
            if _ro is not None:
                _mso = _mo if _mo is not None else _ro**2
                _m = {"mean_rmse_op": _ro, "mean_mse_op": _mso}
                if _rb is not None:
                    _m.update({
                        "mean_rmse_bnd": _rb,
                        "mean_mse_bnd": _rb**2,
                        "mean_rmse_op_plus_bnd": _ro + _rb,
                        "mean_mse_op_plus_bnd": _mso + _rb**2,
                        # Objective = rmse_op + rmse_bnd (what Optuna minimized)
                        "mean_objective_rmse_plus_brmse": _ro + _rb,
                    })
                experiment.log_metrics(_m)
    else:
        print(f"WARNING: Results CSV not found at {RESULTS_CSV}")

    print(f"Comet ML experiment '{EXPERIMENT_NAME}' logged successfully.")
    print(f"View at: https://www.comet.com/{COMET_WORKSPACE}/{COMET_PROJECT_NAME}")

except Exception as _e:
    import traceback
    print(f"Comet ML logging failed: {_e}")
    traceback.print_exc()
finally:
    experiment.end()

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.


COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


COMET INFO: Experiment is live on comet.com https://www.comet.com/florentiner/optuna-rl-kaggle/f8f30e3d53934b86b329721f8c9011ea



COMET INFO: Couldn't find a Git repository in '/Users/danilezov/work/optuna_exp/PINNacle' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


COMET INFO: ---------------------------------------------------------------------------------------


...

COMET INFO:     notebook            : 1


COMET INFO:     source_code         : 1


COMET INFO: 


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.


Comet ML experiment 'burgers_1d_continuous' logged successfully.
View at: https://www.comet.com/florentiner/optuna-rl-kaggle


In [13]:
# Optional: run GPU benchmark to understand per-PDE memory usage
# Uncomment and run this cell separately if needed
# !python experiments/optuna_multi_pde/benchmark_pde_gpu.py --pdes {PDE_NAME} --steps 200